In [0]:
%pip install scikit-learn

In [0]:
import numpy as np
import pandas as pd
import mlflow
from pyspark.sql import functions as F
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

spark.conf.set("spark.sql.session.timeZone", "Asia/Singapore")

# 1. One row per station per day: peak and average WBGT (only near-complete days)
daily = (spark.table("workspace.heatcheck.silver_wbgt")
    .withColumn("day", F.to_date("reading_time"))
    .groupBy("station_id", "day")
    .agg(F.max("wbgt").alias("peak"), F.avg("wbgt").alias("mean_today"), F.count("*").alias("n"))
    .filter("n >= 80")
    .toPandas())

daily["day"] = pd.to_datetime(daily["day"])
daily = daily.sort_values(["station_id", "day"]).reset_index(drop=True)
g = daily.groupby("station_id")
daily["peak_lag1"] = g["peak"].shift(1)
daily["peak_3d"] = g["peak"].transform(lambda s: s.rolling(3).mean())
daily["peak_7d"] = g["peak"].transform(lambda s: s.rolling(7).mean())
daily["target"] = g["peak"].shift(-1)
daily["target_day"] = g["day"].shift(-1)
daily = daily[(daily["target_day"] - daily["day"]).dt.days == 1]
doy = daily["day"].dt.dayofyear
daily["doy_sin"] = np.sin(2 * np.pi * doy / 365.25)
daily["doy_cos"] = np.cos(2 * np.pi * doy / 365.25)
daily = daily.dropna(subset=["peak_lag1", "peak_7d", "target"])

# 2. Train on the past, test on the most recent months
CUT = pd.Timestamp("2026-04-01")
train = daily[daily["day"] < CUT].copy()
test = daily[daily["day"] >= CUT].copy()
station_avg = train.groupby("station_id")["peak"].mean()
train["station_avg"] = train["station_id"].map(station_avg)
test["station_avg"] = test["station_id"].map(station_avg)
test = test.dropna(subset=["station_avg"])

FEATURES = ["peak", "mean_today", "peak_lag1", "peak_3d", "peak_7d", "doy_sin", "doy_cos", "station_avg"]
model = HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05, max_leaf_nodes=15, random_state=42)
model.fit(train[FEATURES], train["target"])
pred = model.predict(test[FEATURES])

# 3. Compare against simple baselines
y = test["target"].values
mae_persist = mean_absolute_error(y, test["peak"])
mae_7d = mean_absolute_error(y, test["peak_7d"])
mae_model = mean_absolute_error(y, pred)

def level(x):
    x = np.asarray(x)
    return np.where(x >= 33, "High", np.where(x >= 31, "Moderate", "Low"))

acc_persist = (level(test["peak"]) == level(y)).mean()
acc_model = (level(pred) == level(y)).mean()
hot = y >= 31
catch_persist = (np.asarray(test["peak"])[hot] >= 31).mean() if hot.any() else float("nan")
catch_model = (pred[hot] >= 31).mean() if hot.any() else float("nan")
improvement = (mae_persist - mae_model) / mae_persist

# 4. Track the experiment in MLflow
with mlflow.start_run(run_name="next_day_peak_wbgt_v0"):
    mlflow.log_params({"model": "HistGradientBoostingRegressor", "features": ",".join(FEATURES),
                       "test_from": str(CUT.date()), "train_rows": len(train), "test_rows": len(test)})
    mlflow.log_metrics({"mae_model": mae_model, "mae_persistence": mae_persist, "mae_7day_avg": mae_7d,
                        "level_accuracy_model": acc_model, "level_accuracy_persistence": acc_persist,
                        "improvement_vs_persistence": improvement})
    try:
        mlflow.sklearn.log_model(model, "model", input_example=train[FEATURES].head(3))
    except Exception as e:
        print("Model not saved as artifact (metrics still logged):", e)

# 5. Results
start = test["day"].min().strftime("%d %b %Y")
end = test["day"].max().strftime("%d %b %Y")
print("Test period:", start, "to", end, "|", len(test), "station-days | trained on", len(train))
print("")
print("Average error in tomorrow's peak WBGT (lower is better):")
print("  Tomorrow = today :", round(mae_persist, 2), "C")
print("  7-day average    :", round(mae_7d, 2), "C")
print("  HeatCheck model  :", round(mae_model, 2), "C")
print("  Improvement vs tomorrow = today:", str(round(improvement * 100)) + "%")
print("")
print("Heat level correct (Low / Moderate / High):")
print("  Tomorrow = today:", str(round(acc_persist * 100)) + "%")
print("  HeatCheck model :", str(round(acc_model * 100)) + "%")
print("")
print("Moderate-or-higher days caught:")
print("  Tomorrow = today:", str(round(catch_persist * 100)) + "%")
print("  HeatCheck model :", str(round(catch_model * 100)) + "%")